# PREPARACIÓN DE DATOS PARA EL MODELADO

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import style

import seaborn as sns


%matplotlib inline

In [ ]:
import warnings
warnings.filterwarnings("ignore")

style.use('ggplot') or plt.style.use('ggplot')

## CODIFICACIÓN DE VARIABLES CATEGÓRICAS

**Me creo un dataframe con columnas numéricas y categóricas, y target categórico:**

In [ ]:
np.random.seed(42)

df = pd.DataFrame({'altura': np.random.normal(170,10, size=5), 'peso': np.random.normal(80,8,size=5), \
                   'sexo': list('MMFMF'), 'raza': ['caucasica','afro','asiatica','caucasica','afro'], \
                   'valoracion': ['mala', 'buena', 'regular', 'buena', 'mala'], \
                   'respuesta': ['si','no','si','si','no']})

df

In [ ]:
df.dtypes

**Es buena idea "etiquetar y separar" por tipos:**

In [ ]:
target = 'respuesta'

In [ ]:
var_num = df.select_dtypes(exclude=['object']).columns.to_list()

var_num

In [ ]:
var_cat = df.select_dtypes(include=['object']).columns.to_list()

var_cat

In [ ]:
var_cat.remove(target)

In [ ]:
var_cat

**Separo atributos ("X") y el target ("y"):**

In [ ]:
X = df.drop(columns=target)

y = df[target]

X.shape,y.shape, type(X), type(y)

### ONE HOT ENCODER

Estoy codificando a número, lo puedo aplicar antes de la partición TRAIN-TEST sin problema, a todo "X":

In [ ]:
from sklearn.preprocessing import OneHotEncoder


In [ ]:
import sklearn
sklearn.__version__

In [ ]:
OHE = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprx_data_cat_ohe = OHE.fit_transform(X[['sexo','raza']])

col_data_cat_ohe = OHE.get_feature_names_out()

preprx_data_cat_ohe

In [ ]:
col_data_cat_ohe

In [ ]:
X_cat_ohe = pd.DataFrame(data=preprx_data_cat_ohe, columns=col_data_cat_ohe)

X_cat_ohe

In [ ]:
df[['sexo','raza']]

### LABEL ENCODER

La podemos dejar en formato array de numpy:

In [ ]:
from sklearn.preprocessing import LabelEncoder


In [ ]:
target_encoder = LabelEncoder()

y_num = target_encoder.fit_transform(y)

y_num

In [ ]:
target_encoder.classes_

In [ ]:
y.values

In [ ]:
columna_target = pd.Series(data=y_num)

columna_target

### ORDINAL ENCODER

Nos queda la columna "valoracion":

In [ ]:
from sklearn.preprocessing import OrdinalEncoder


In [ ]:
ord_encoder = OrdinalEncoder(categories=[['mala', 'regular', 'buena']])

valoracion_codificada = ord_encoder.fit_transform(X.valoracion.values.reshape(-1,1))

valoracion_codificada

In [ ]:
X.valoracion.values

In [ ]:
ord_encoder.categories_

## ESTANDARIZACIÓN / NORMALIZACIÓN

**NOTA**:
Estas operaciones debo hacerlas DESPUÉS de la partición TRAIN-TEST, para evitar el fenómeno de "fuga"  (leakage).
Como aquí estamos presentando los métodos, prescindiremos por ahora de ese paso

### STANDAR SCALER

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X1_prx_num = scaler.fit_transform(X[var_num])

X1_prx_num

In [ ]:
df_X1 = pd.DataFrame(data=X1_prx_num,columns=scaler.get_feature_names_out(), index = X[var_num].index)

df_X1

In [ ]:
df_X1.peso.mean()

### MINMAX SCALER

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X2_prx_num = scaler.fit_transform(X[var_num])

X2_prx_num

In [ ]:
df_X2 = pd.DataFrame(data=X2_prx_num,columns=scaler.get_feature_names_out(), index = X[var_num].index)

df_X2

### ROBUST SCALER

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X3_prx_num = scaler.fit_transform(X[var_num])

X3_prx_num

In [ ]:
df_X3 = pd.DataFrame(data=X3_prx_num,columns=scaler.get_feature_names_out(), index = X[var_num].index)

df_X3

**Finalmente comparamos los 3:**

In [ ]:
df_X1

In [ ]:
df_X2

In [ ]:
df_X3

In [ ]:
df_X3.shape, X_cat_ohe.shape, valoracion_codificada.shape

In [ ]:
X_prx = pd.concat([df_X3, X_cat_ohe, \
                   pd.DataFrame(data=valoracion_codificada, columns=['valoracion'])], \
                  axis=1)

X_prx

In [ ]:
y_num.shape

In [ ]:
df_recuperado = pd.concat([df_X3, X_cat_ohe, \
                           pd.DataFrame(data=valoracion_codificada, columns=['valoracion']), \
                           pd.DataFrame(data=y_num.reshape(5,-1), columns=['respuesta'])], \
                          axis=1)

df_recuperado